# 줄바꿈 정리 → 연결표 → 재임베딩 (Colab GPU)

DB **main**의 `rag_unwrap.py`로 조문·부칙의 끊긴 낱말 줄바꿈을 고치고, 그 뒤 단계(연결표 다시 만들기 · BGE-M3 재임베딩 · 검사)를 한 번에 해요.
노션 가이드 "규정 조각 줄바꿈(끊긴 낱말) 정리 가이드"의 할 일 중 **데이터 만들기**까지예요.

**실행:** 런타임 → 런타임 유형 변경 → **GPU (T4 이상)** → 모두 실행. 10분 안팎.

| 단계 | 하는 일 | 결과 |
| --- | --- | --- |
| 1 | DB 저장소 받기 (main) | |
| 2 | 고치기 전 숫자 세기 | 조각 수 · 끊긴 조각 · 연결표 행 수 |
| 3 | `test_rag_unwrap.py` → `rag_unwrap.py` | `data/hang/chunks.jsonl` · `articles.jsonl` |
| 4 | `rag_appendix_links.py` · `rag_article_refs.py` | `data/hang/*_links.json` · `article_refs.json` · `deploy/initdb/07`·`10` |
| 5 | BGE-M3 재임베딩 (검색 조각 전부, fp16, 최대 4096토큰) | `data/hang/embeddings.npy` · `embedding_ids.json` · `embedding_texts.json` · `embedding_meta.json` |
| 6 | 검사 (하나라도 어긋나면 멈춤) | |
| 7 | README에 `rag_unwrap.py` 한 줄 추가 | `README.md` |
| 8 | 바뀐 파일 zip 내려받기 + Drive | `unwrap_reembed_*.zip` |

**끝나면:** zip을 풀어 DB 저장소에 덮어쓰고, 웹 업로드 말고 **브랜치 + PR**로 올려요 (맨 아래 명령어).

In [ ]:
!pip -q install -U "sentence-transformers>=3.4" kiwipiepy pytest

## 1. DB 저장소 받기

In [ ]:
import os, sys, json, re, subprocess, shutil, time, zipfile, copy
from datetime import datetime
from pathlib import Path

BRANCH = "main"                                   # 다른 브랜치에서 하려면 여기만 바꿈
ROOT = Path(os.environ.get("LOCAL_ROOT", "/content"))
DB = ROOT / "DB"
if "LOCAL_ROOT" not in os.environ:                # Colab: 매번 새로 받음
    if DB.exists():
        shutil.rmtree(DB)
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/Jaeyong-s-HR-Chatbot/DB.git", str(DB)], check=True)
os.chdir(DB)
COMMIT = subprocess.run(["git", "log", "-1", "--format=%h %ad %s", "--date=short"],
                        capture_output=True, text=True, encoding="utf-8").stdout.strip()
print("DB", BRANCH, COMMIT)
for f in ("rag_unwrap.py", "test_rag_unwrap.py", "rag_appendix_links.py", "rag_article_refs.py"):
    assert Path(f).exists(), f"{BRANCH} 에 {f} 가 없어요"
HANG = Path("data/hang")

## 2. 고치기 전 숫자

In [ ]:
BRK = re.compile(r"[가-힣]\n[가-힣]")              # 가이드의 확인 기준: 한글 ↔ 한글 사이 줄바꿈
SQL_ROWS = {"07 연결표(별표·서식)": "deploy/initdb/07_appendix_links.sql", "10 인용표": "deploy/initdb/10_article_refs.sql"}

def jl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

def sql_rows(p):                                  # INSERT 값 줄 수 = 표 행 수
    return sum(1 for l in open(p, encoding="utf-8") if l.startswith("('"))

def stats():
    ch, ar = jl(HANG / "chunks.jsonl"), jl(HANG / "articles.jsonl")
    tgt = [c for c in ch if c.get("searchable") and c["chunk_type"] in ("조문", "부칙")]
    s = {"조각": len(ch), "검색 조각": sum(bool(c.get("searchable")) for c in ch), "조문 원문": len(ar),
         "끊긴 조문·부칙 조각(검색)": sum(bool(BRK.search(c["text"])) for c in tgt)}
    s |= {k: sql_rows(v) for k, v in SQL_ROWS.items()}
    return s, ch, ar

BEFORE, CH0, AR0 = stats()
CH0, AR0 = copy.deepcopy(CH0), copy.deepcopy(AR0)
REFS0 = json.load(open(HANG / "article_refs.json", encoding="utf-8"))["refs"]
for k, v in BEFORE.items():
    print(f"{k:24s} {v:,}")

## 3. 줄바꿈 정리 (`rag_unwrap.py`)

In [ ]:
def run(*args):
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True, encoding="utf-8", errors="replace",
                       env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    print(r.stdout[-3000:], r.stderr[-3000:])
    assert r.returncode == 0, f"{args[0]} 실패 (위 출력)"

run("-m", "pytest", "test_rag_unwrap.py", "-q")    # 규칙 시험 먼저
run("rag_unwrap.py", "--show", "10")               # data/hang/chunks.jsonl · articles.jsonl 제자리 고침

## 4. 연결표 다시 만들기

In [ ]:
run("rag_appendix_links.py")                      # 조↔별표 · 조↔별지 서식 → 07 SQL
run("rag_article_refs.py")                        # 조↔조 인용 → 10 SQL
AFTER_TEXT, CH1, AR1 = stats()
for k in BEFORE:
    print(f"{k:24s} {BEFORE[k]:>7,} → {AFTER_TEXT[k]:>7,}")

## 5. BGE-M3 재임베딩

검색 조각 **전부** 새로 계산해요 (GPU면 몇 분). 입력 형식은 `rag_load_hang.py`의 `doc_text`와 같아야 DB에 넣을 때 재사용돼요: `규정명 조문 제목\n본문`.

In [ ]:
import numpy as np, torch
from sentence_transformers import SentenceTransformer

def doc_text(c):                                  # rag_load_hang.py · rag_embed_hang_colab.ipynb 와 같음 (바꾸면 재사용 안 됨)
    head = f"{c['doc_name']} {c['article']} {c['article_title']}".strip()
    return f"{head}\n{c['text']}"

target = [c for c in CH1 if c.get("searchable")]
texts = [doc_text(c) for c in target]
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"검색 조각 {len(target):,}개 · {device}" + ("" if device == "cuda" else "  ← GPU가 아니에요. 런타임 유형을 GPU로 바꾸세요"))
model = SentenceTransformer("BAAI/bge-m3", device=device)
model.max_seq_length = 4096                       # 긴 별표(최대 약 3,200토큰)가 잘리지 않게 (#27과 같음)
if device == "cuda":
    model.half()
order = sorted(range(len(texts)), key=lambda i: len(texts[i]))   # 길이 비슷한 것끼리 묶어 빠르게
t0 = time.time()
vec = model.encode([texts[i] for i in order], batch_size=32, normalize_embeddings=True,
                   show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
emb = np.empty_like(vec)
emb[order] = vec                                  # 원래 순서로 되돌림
print(emb.shape, f"{time.time() - t0:.0f}초")

np.save(HANG / "embeddings.npy", emb)
json.dump([c["chunk_id"] for c in target], open(HANG / "embedding_ids.json", "w", encoding="utf-8"))
json.dump(texts, open(HANG / "embedding_texts.json", "w", encoding="utf-8"), ensure_ascii=False)
json.dump({"model": "BAAI/bge-m3", "dim": int(emb.shape[1]), "count": len(target), "normalized": True,
           "input": "규정명 조문 제목\\n본문", "max_seq_length": 4096,
           "device": f"{device} (colab, fp16)" if device == "cuda" else device,
           "db_commit": COMMIT, "after": "rag_unwrap.py"},
          open(HANG / "embedding_meta.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

## 6. 검사 (하나라도 어긋나면 멈춤)

In [ ]:
problems = []
def check(ok, msg):
    print(("✅ " if ok else "❌ ") + msg)
    if not ok:
        problems.append(msg)

# 조각·id: 글만 바뀌고 나머지는 그대로
check(AFTER_TEXT["조각"] == BEFORE["조각"] and AFTER_TEXT["검색 조각"] == BEFORE["검색 조각"],
      f"조각 수 그대로 ({BEFORE['조각']:,} / 검색 {BEFORE['검색 조각']:,})")
check([c["chunk_id"] for c in CH1] == [c["chunk_id"] for c in CH0], "chunk_id 순서까지 그대로")
other = [c1["chunk_id"] for c0, c1 in zip(CH0, CH1) if {k: v for k, v in c0.items() if k != "text"} != {k: v for k, v in c1.items() if k != "text"}]
check(not other, f"text 말고 다른 칸은 그대로 (다른 조각 {len(other)}개)")
touched = [c1 for c0, c1 in zip(CH0, CH1) if c0["text"] != c1["text"]]
check(all(c["chunk_type"] in ("조문", "부칙") for c in touched), f"바뀐 조각 {len(touched):,}개 모두 조문·부칙")
check(all(re.sub(r"\s", "", c0["text"]) == re.sub(r"\s", "", c1["text"]) for c0, c1 in zip(CH0, CH1)),
      "공백·줄바꿈 말고 글자는 하나도 안 바뀜")
check([a["article_id"] for a in AR1] == [a["article_id"] for a in AR0], "조문 원문 article_id 그대로")

# 줄바꿈
check(AFTER_TEXT["끊긴 조문·부칙 조각(검색)"] < 100,
      f"끊긴 조문·부칙 조각 {BEFORE['끊긴 조문·부칙 조각(검색)']:,} → {AFTER_TEXT['끊긴 조문·부칙 조각(검색)']:,} (목표 100 아래)")

# 연결표: 조↔별표·서식은 줄면 안 됨
k = "07 연결표(별표·서식)"
check(AFTER_TEXT[k] >= BEFORE[k], f"{k} 행 수 줄지 않음 ({BEFORE[k]:,} → {AFTER_TEXT[k]:,})")
# 조↔조 인용은 바뀔 수 있음: 「국민건강보\n험법」 제52조처럼 규정 이름이 줄바꿈으로 끊겨 있으면
# 예전엔 이름을 못 알아봐서 같은 규정의 제52조로 잘못 연결했음 → 고치면 다른 규정으로 바로잡히거나(+), DB에 없는 법이라 빠짐(-)
key = lambda r: (r["from"], r["to"], r["kind"])
REFS1 = json.load(open(HANG / "article_refs.json", encoding="utf-8"))["refs"]
lost = sorted(set(map(key, REFS0)) - set(map(key, REFS1)))
new = sorted(set(map(key, REFS1)) - set(map(key, REFS0)))
print(f"조↔조 인용 {len(REFS0):,} → {len(REFS1):,} (빠짐 {len(lost)}, 생김 {len(new)}) — 아래가 맞는지 눈으로 확인")
for f, t, kd in lost:
    print(f"   - {f} → {t} ({kd})")
for f, t, kd in new:
    print(f"   + {f} → {t} ({kd})")
check(len(lost) <= 50, "인용 연결이 크게 줄지 않음 (빠짐 50개 이하)")
same_doc_lost = [x for x in lost if x[0].split(":")[0] == x[1].split(":")[0]]
check(len(same_doc_lost) == len(lost), f"빠진 인용은 모두 '같은 규정 안' 연결 (잘못 붙었던 것) — {len(same_doc_lost)}/{len(lost)}")

# 임베딩
ids = json.load(open(HANG / "embedding_ids.json", encoding="utf-8"))
arr = np.load(HANG / "embeddings.npy")
check(ids == [c["chunk_id"] for c in CH1 if c.get("searchable")] and len(ids) == len(set(ids)),
      f"임베딩 id = 검색 조각 전부 ({len(ids):,}개, 중복 없음)")
check(arr.shape == (len(ids), 1024) and bool(np.isfinite(arr).all()), f"임베딩 모양 {arr.shape}, NaN 없음")
n = np.linalg.norm(arr, axis=1)
check(bool(np.allclose(n, 1, atol=1e-2)), f"정규화됨 (길이 {n.min():.3f}~{n.max():.3f})")

# 검색 맛보기 (벡터만)
for q in ("가족수당은 누구에게 얼마나 줘요?", "결혼하면 휴가 며칠이야?", "연차수당은 언제 지급돼요?"):
    qv = model.encode([q], normalize_embeddings=True, convert_to_numpy=True)[0].astype(np.float32)
    top = np.argsort(-(arr @ qv))[:3]
    print(f"  {q} → " + " · ".join(f"{target[i]['article_id']}" for i in top))

assert not problems, f"검사 실패 {len(problems)}건 — 위 ❌ 확인"
print("\n검사 모두 통과")

## 7. README에 한 줄 추가

In [ ]:
readme = Path("README.md").read_text(encoding="utf-8")
LINE = ".venv\\Scripts\\python rag_unwrap.py           # 조문·부칙 끊긴 낱말 줄바꿈 정리 (rechunk 다음, 연결표·임베딩 앞)\n"
anchor = next((l for l in readme.splitlines(keepends=True) if l.lstrip().startswith(".venv\\Scripts\\python rag_rechunk.py")), None)
if "rag_unwrap.py" in readme:
    print("README에 이미 있음")
elif anchor:
    Path("README.md").write_text(readme.replace(anchor, anchor + LINE, 1), encoding="utf-8")
    print("README 추가:", LINE.strip())
else:
    print("⚠️ README에서 rag_rechunk.py 실행 줄을 못 찾음 → 손으로 추가하세요")

## 8. 바뀐 파일 내려받기

In [ ]:
changed = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout.split("\n")
changed = sorted(l[3:].strip().strip('"') for l in changed if l.strip())
print("바뀐 파일:"); print("\n".join("  " + f for f in changed))
stamp = datetime.now().strftime("%Y%m%d_%H%M")
ZIP = ROOT / f"unwrap_reembed_{stamp}.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in changed:
        z.write(f)
    z.writestr("RESULT.txt", f"DB {BRANCH} {COMMIT}\n" + "\n".join(f"{k}: {BEFORE[k]} -> {AFTER_TEXT[k]}" for k in BEFORE) + "\n")
print(f"\n저장: {ZIP} ({ZIP.stat().st_size / 1e6:.1f}MB)")
try:
    from google.colab import files, drive
    files.download(str(ZIP))
    drive.mount("/content/drive")
    dst = Path("/content/drive/MyDrive/hr_chatbot_eval"); dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(ZIP, dst); print("Drive에도 저장:", dst / ZIP.name)
except Exception as e:
    print(f"내려받기·Drive 건너뜀 ({type(e).__name__})")

## 9. 저장소에 올리기 (내 컴퓨터에서)

zip을 DB 저장소 폴더에 풀어서 **덮어쓰기** 한 뒤 (zip 안 경로가 저장소 경로와 같아요):

```powershell
git switch main
git pull
git switch -c fix/unwrap-reembed
# 여기서 zip 풀어 덮어쓰기
git add -A data/hang deploy/initdb README.md
git commit -m "fix(rag): 조문·부칙 끊긴 낱말 줄바꿈 정리 결과 + 연결표 + 재임베딩"
git push -u origin fix/unwrap-reembed
```

GitHub에서 PR을 만들어요. **웹 업로드(Add files via upload)는 쓰지 마세요** — 파일 하나 25MB 제한이 있고 PR 검토가 안 돼요.
로컬 DB에 넣을 때는 `rag_load_hang.py`가 임베딩을 다시 계산하지 않고(재사용) 넣기만 해요. 그다음 `deploy/initdb/06~10` SQL.